# 00 — QA propre du pricer de produits structurés

**Objectif** : exécuter un contrôle qualité reproductible avant clean-up et rendu final.

Ce notebook vérifie :

1. les imports et les tests unitaires ;
2. le bootstrapping de courbe de taux dépôt / FRA / swap ;
3. les conventions day count / business day et la cohérence ZC ;
4. la calibration implied volatility ;
5. les surfaces SVI / SSVI et les diagnostics d'arbitrage ;
6. le repricing d'un panel de vanilles marché ;
7. l'ingestion et l'agrégation de l'inventaire portefeuille ;
8. une synthèse finale `PASS / WARN / FAIL` exportée dans `reports/qa/`.

> Lecture recommandée : un `WARN` n'est pas forcément bloquant. Il signale un point à justifier dans le rapport, typiquement une surface SVI bruitée sur données marché gratuites.

**Limite générale des validations** : ce notebook finalise les fondations marché avec les données disponibles dans le cours. Il ne télécharge pas de nouvelles données. Les validations qui nécessiteraient des quotes dépôt/FRA/swap réelles ou un panel options multi-dates sont donc séparées des validations synthétiques du moteur.

In [12]:
from __future__ import annotations

import math
import os
import subprocess
import sys
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display, Markdown
except Exception:  # pragma: no cover
    display = print
    Markdown = lambda x: x

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
np.set_printoptions(precision=6, suppress=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---------------------------------------------------------------------------
# Résolution robuste du root projet
# ---------------------------------------------------------------------------

def find_project_root() -> Path:
    """Find the repository root containing src/ and tests/."""
    candidates: list[Path] = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    # Useful when the notebook is executed from the uploaded sandbox or a copied folder.
    candidates.extend([Path('/mnt/data'), Path('/mnt/data').parent])

    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'tests').exists():
            return candidate
        # Fallback for the uploaded flat-file context: not used in the final repo, but useful for review.
        if (candidate / 'black_scholes.py').exists() and (candidate / 'test_vanilla_pricing.py').exists():
            return candidate

    raise FileNotFoundError("Impossible de trouver le root projet. Lance ce notebook depuis le repo ou depuis notebooks/.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QA_OUTPUT_DIR = PROJECT_ROOT / 'reports' / 'qa'
QA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Paramètres de validation marché : volontairement plus tolérants que les tests synthétiques.
QA_RATE = 0.03
QA_DIVIDEND_YIELD = 0.00
MAX_VOL_QUOTES = 3000
TRAIN_FRACTION = 0.80

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"QA_OUTPUT_DIR = {QA_OUTPUT_DIR}")

PROJECT_ROOT = C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer
QA_OUTPUT_DIR = C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa


In [13]:
# ---------------------------------------------------------------------------
# Mini framework QA : chaque contrôle alimente une table finale.
# ---------------------------------------------------------------------------

qa_rows: list[dict[str, Any]] = []


def _short(value: Any, max_len: int = 160) -> str:
    text = str(value)
    return text if len(text) <= max_len else text[: max_len - 3] + '...'


def record(
    section: str,
    check: str,
    observed: Any,
    expected: Any,
    status: str,
    details: str = "",
) -> None:
    status = status.upper()
    if status not in {"PASS", "WARN", "FAIL"}:
        raise ValueError("status must be PASS, WARN or FAIL")
    qa_rows.append(
        {
            "section": section,
            "check": check,
            "observed": _short(observed),
            "expected": _short(expected),
            "status": status,
            "details": _short(details, max_len=500),
        }
    )


def check_true(
    section: str,
    check: str,
    condition: bool,
    *,
    observed: Any | None = None,
    expected: Any = True,
    fail_if_false: bool = True,
    details: str = "",
) -> None:
    record(
        section=section,
        check=check,
        observed=condition if observed is None else observed,
        expected=expected,
        status="PASS" if condition else ("FAIL" if fail_if_false else "WARN"),
        details=details,
    )


def record_exception(section: str, check: str, exc: BaseException, *, fail: bool = True) -> None:
    record(
        section=section,
        check=check,
        observed=f"{type(exc).__name__}: {exc}",
        expected="no exception",
        status="FAIL" if fail else "WARN",
        details=traceback.format_exc(limit=2),
    )


def qa_table() -> pd.DataFrame:
    return pd.DataFrame(qa_rows, columns=["section", "check", "observed", "expected", "status", "details"])


def display_qa(section: str | None = None) -> None:
    frame = qa_table()
    if section is not None and not frame.empty:
        frame = frame[frame["section"] == section]
    display(frame)


def save_frame(frame: pd.DataFrame, name: str) -> Path:
    path = QA_OUTPUT_DIR / name
    frame.to_csv(path, index=False)
    print(f"Exported: {path}")
    return path


def non_empty_frame(frame: pd.DataFrame | None) -> bool:
    return isinstance(frame, pd.DataFrame) and not frame.empty

In [14]:
# ---------------------------------------------------------------------------
# Imports projet
# ---------------------------------------------------------------------------

try:
    from src.config import ProjectConfig
    from src.conventions.business_day import BusinessCalendar
    from src.conventions.day_count import year_fraction
    from src.market.loaders import load_option_quotes, load_rate_curves
    from src.portfolio.inventory_loader import (
        combine_inventory_sheets,
        inventory_dataset_summary,
        load_inventory_workbook,
    )
    from src.rates.bootstrap import bootstrap_yield_curve
    from src.rates.market_instruments import BootstrapMarket, DepositQuote, FRAQuote, SwapQuote
    from src.rates.yield_curve import YieldCurve
    from src.calibration.implied_vol import (
        calibrate_implied_vol_panel,
        clean_option_panel,
        implied_volatility_from_price,
    )
    from src.calibration.market_validation import MarketErrorThresholds, reprice_vanilla_market_quotes
    from src.calibration.svi import SVIVolSurface, SSVIVolSurface
    from src.market.market_data import MarketData
    from src.models.black_scholes import BlackScholesModel, black_scholes_price_and_greeks
    from src.models.discounting_model import DiscountingModel
    from src.portfolio.book import PortfolioMarketContext, PortfolioValuationEngine
    from src.products.vanilla_option import VanillaOption
    from src.products.zero_coupon_bond import ZeroCouponBond

    CONFIG = ProjectConfig.default(PROJECT_ROOT if (PROJECT_ROOT / 'src').exists() else None)
    check_true("architecture", "project_imports", True, details="Core src imports succeeded.")
except Exception as exc:
    record_exception("architecture", "project_imports", exc, fail=True)
    CONFIG = None

print("Imports projet terminés.")
display_qa("architecture")

Imports projet terminés.


,section,check,observed,expected,status,details
0,architecture,project_imports,True,True,PASS,Core src imports succeeded.


## 1. Tests unitaires

Cette cellule lance `pytest -q`. Elle est volontairement séparée des contrôles métier : les tests unitaires doivent rester la première barrière de non-régression.

In [15]:
try:
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
        timeout=180,
    )
    output_tail = (result.stdout + "\n" + result.stderr)[-4000:]
    print(output_tail)
    check_true(
        "unit_tests",
        "pytest_q",
        result.returncode == 0,
        observed=f"returncode={result.returncode}",
        expected="returncode=0",
        fail_if_false=True,
        details=output_tail,
    )
except Exception as exc:
    record_exception("unit_tests", "pytest_q", exc, fail=True)

display_qa("unit_tests")

................................................                         [100%]
48 passed in 54.49s




,section,check,observed,expected,status,details
1,unit_tests,pytest_q,returncode=0,returncode=0,PASS,[32m.[0m[32m.[0m[32m.[0m[32m.[0m[32m....


## 2. Chargement des données

On charge les datasets si disponibles. Une absence de données est un `WARN`, pas un `FAIL`, car le moteur peut être validé sur fixtures synthétiques.

Commentaires de périmètre :

- les loaders privilégient les copies locales dans `data/raw/`, puis les chemins originaux du cours en fallback ;
- le fichier de taux du cours est un panel de courbes historiques par pays/date/tenor, pas un fichier d'instruments dépôt/FRA/swap ;
- le fichier options contient des prix mais pas une colonne d'IV exploitable, donc les volatilités implicites sont recalculées depuis les prix ;
- l'inventaire est chargé en entier, mais tous les produits chargés ne sont pas encore dans le scope de pricing.

In [16]:
rate_curves = pd.DataFrame()
option_quotes = pd.DataFrame()
inventory_by_sheet: dict[str, pd.DataFrame] = {}
inventory_all = pd.DataFrame()

try:
    if CONFIG is None:
        raise RuntimeError("ProjectConfig unavailable")
    rate_curves = load_rate_curves(config=CONFIG)
    check_true("data", "rate_curves_loaded", len(rate_curves) > 0, observed=len(rate_curves), expected="> 0", fail_if_false=False)
    display(rate_curves.head())
except Exception as exc:
    record_exception("data", "rate_curves_loaded", exc, fail=False)

try:
    if CONFIG is None:
        raise RuntimeError("ProjectConfig unavailable")
    option_quotes = load_option_quotes(config=CONFIG)
    check_true("data", "option_quotes_loaded", len(option_quotes) > 0, observed=len(option_quotes), expected="> 0", fail_if_false=False)
    display(option_quotes.head())
except Exception as exc:
    record_exception("data", "option_quotes_loaded", exc, fail=False)

try:
    if CONFIG is None:
        raise RuntimeError("ProjectConfig unavailable")
    inventory_by_sheet = load_inventory_workbook(config=CONFIG)
    inventory_all = combine_inventory_sheets(inventory_by_sheet)
    check_true("data", "inventory_loaded", len(inventory_all) > 0, observed=len(inventory_all), expected="> 0", fail_if_false=False)
    display(inventory_dataset_summary(inventory_by_sheet))
except Exception as exc:
    record_exception("data", "inventory_loaded", exc, fail=False)

display_qa("data")

,country,curve_tenor,curve_tenor_years,observation_date,rate_percent,rate_decimal
0,France,10Y,10.0,1986-01-02,9.989,0.09989
1,France,10Y,10.0,1986-01-03,9.993,0.09993
2,France,10Y,10.0,1986-01-06,9.997,0.09997
3,France,10Y,10.0,1986-01-07,10.002,0.10002
4,France,10Y,10.0,1986-01-08,10.006,0.10006


,contract_symbol,ticker,underlying,option_type,valuation_date,expiration_date,time_to_maturity_days,time_to_maturity_years,strike,bid,mid,ask,last,underlying_price,implied_volatility,intrinsic_value,extrinsic_value,bid_size,ask_size,open_interest,volume,in_the_money,first_traded_at,last_updated_at,delta,gamma,theta,vega,rho
0,AAPL260320C00090000,AAPL,AAPL,call,2026-03-02,2026-03-20,18,0.049281,90.0,173.75,175.2,176.65,175.67,264.72,<NA>,174.72,0.48,105,110,59,0,True,2025-04-08 13:30:00,2026-03-02 21:00:00,<NA>,<NA>,<NA>,<NA>,<NA>
1,AAPL260320C00095000,AAPL,AAPL,call,2026-03-02,2026-03-20,18,0.049281,95.0,168.4,170.125,171.85,165.52,264.72,<NA>,169.72,0.405,100,100,1,0,True,2025-04-08 13:30:00,2026-03-02 21:00:00,<NA>,<NA>,<NA>,<NA>,<NA>
2,AAPL260320C00100000,AAPL,AAPL,call,2026-03-02,2026-03-20,18,0.049281,100.0,163.2,165.025,166.85,164.7,264.72,<NA>,164.72,0.305,100,100,54,0,True,2025-04-08 13:30:00,2026-03-02 21:00:00,<NA>,<NA>,<NA>,<NA>,<NA>
3,AAPL260320C00105000,AAPL,AAPL,call,2026-03-02,2026-03-20,18,0.049281,105.0,158.8,160.325,161.85,143.15,264.72,<NA>,159.72,0.605,105,107,27,0,True,2025-04-08 13:30:00,2026-03-02 21:00:00,<NA>,<NA>,<NA>,<NA>,<NA>
4,AAPL260320C00110000,AAPL,AAPL,call,2026-03-02,2026-03-20,18,0.049281,110.0,153.8,155.325,156.85,154.77,264.72,<NA>,154.72,0.605,209,237,355,0,True,2025-03-18 13:30:00,2026-03-02 21:00:00,<NA>,<NA>,<NA>,<NA>,<NA>


,sheet_name,rows,columns,min_date,max_date,missing_cells
0,autocalls,96,9,2026-02-27,2036-01-31,0
1,options,6,14,2026-02-27,2026-09-30,13
2,structured_notes,4,14,2026-02-27,2028-12-31,10
3,swaps,5,12,2026-02-27,2050-12-31,6


,section,check,observed,expected,status,details
2,data,rate_curves_loaded,357978,> 0,PASS,
3,data,option_quotes_loaded,10456,> 0,PASS,
4,data,inventory_loaded,111,> 0,PASS,


## 3. QA taux : bootstrapping dépôt / FRA / swap et ZC

Ce contrôle est volontairement synthétique, car il teste le moteur de bootstrapping indépendamment de la qualité du fichier de marché. Le fichier de taux disponible dans le cours permet de valider la construction d'une courbe ZC historique, la sélection pays/date et les discount factors, mais il ne contient pas les quotes dépôt/FRA/swap brutes nécessaires pour prouver un bootstrap marché complet sur données réelles.

Lecture des résultats :

- `bootstrap_*` valide l'algorithme dépôt/FRA/swap sur un jeu de quotes contrôlé ;
- `historical_curve_*` valide que les courbes historiques du cours sont ingérées et converties en courbe de discounting ;
- cette séparation est volontaire et doit être mentionnée dans le rapport.

In [17]:
bootstrap_result = None

try:
    # Limite des données de cours : le fichier de taux est déjà une courbe historique,
    # pas un set de quotes dépôt/FRA/swap. On valide donc le bootstrap sur quotes
    # contrôlées, puis le loader de courbe historique dans le bloc suivant.
    market = BootstrapMarket(
        valuation_date=pd.Timestamp("2026-04-27"),
        calendar=BusinessCalendar(),
        deposits=(
            DepositQuote("1M", 0.0300),
            DepositQuote("3M", 0.0310),
            DepositQuote("6M", 0.0320),
        ),
        fras=(
            FRAQuote("6M", "9M", 0.0330),
            FRAQuote("9M", "12M", 0.0340),
        ),
        swaps=(
            SwapQuote("2Y", 0.0350),
            SwapQuote("3Y", 0.0360),
            SwapQuote("5Y", 0.0370),
        ),
        name="qa_synthetic_deposit_fra_swap_curve",
    )
    bootstrap_result = bootstrap_yield_curve(market)
    points = bootstrap_result.points
    checks = bootstrap_result.instrument_checks
    diagnostics = bootstrap_result.diagnostics

    display(points)
    display(checks[["instrument_type", "tenor", "quote_rate", "model_rate", "rate_error_bps", "abs_price_error"]])
    print(diagnostics)

    check_true("rates", "bootstrap_point_count", len(points) >= 6, observed=len(points), expected=">= 6")
    check_true("rates", "discount_factors_positive", bool((points["discount_factor"] > 0).all()), observed=points["discount_factor"].min(), expected="> 0")
    check_true("rates", "discount_factors_decreasing", bool(points["discount_factor"].is_monotonic_decreasing), observed=points["discount_factor"].tolist(), expected="monotonic decreasing")
    check_true("rates", "bootstrap_repricing_error_bps", checks["rate_error_bps"].abs().max() < 5.0, observed=float(checks["rate_error_bps"].abs().max()), expected="< 5 bps")
    check_true("rates", "curve_static_arbitrage", bool(diagnostics.get("positive_discount_factors", False)), observed=diagnostics, expected="positive discount factors")

    zc = ZeroCouponBond(product_id="QA-ZC", notional=100.0, maturity=2.0)
    zc_price = DiscountingModel(yield_curve=bootstrap_result.curve).price(zc)
    check_true("rates", "zero_coupon_price_positive", zc_price > 0, observed=zc_price, expected="> 0")

    save_frame(points, "qa_bootstrap_points.csv")
    save_frame(checks, "qa_bootstrap_instrument_checks.csv")
except Exception as exc:
    record_exception("rates", "bootstrap_deposit_fra_swap", exc, fail=True)

# Contrôle additionnel sur la courbe de taux historique si le dataset est disponible.
try:
    if non_empty_frame(rate_curves):
        selected_country = str(rate_curves["country"].dropna().iloc[0])
        market_curve = YieldCurve.from_rate_curves(rate_curves, country=selected_country)
        selected_date = pd.Timestamp(market_curve.name.split()[-1])
        df_1y = market_curve.discount_factor(1.0)
        df_5y = market_curve.discount_factor(5.0)
        check_true("rates", "historical_curve_builds", True, observed=f"{selected_country} {selected_date.date()}", expected="curve object")
        check_true("rates", "historical_curve_df_positive", df_1y > 0 and df_5y > 0, observed=(df_1y, df_5y), expected="> 0")
    else:
        check_true("rates", "historical_curve_builds", False, observed="empty dataset", expected="available dataset", fail_if_false=False)
except Exception as exc:
    record_exception("rates", "historical_curve_builds", exc, fail=False)

display_qa("rates")

,maturity_date,maturity_years,discount_factor,zero_rate,source,quote_rate,pillar_dv01
0,2026-05-27,0.082192,0.997506,0.030379,deposit_1M,0.030,0.000008
1,2026-07-27,0.249315,0.992225,0.031308,deposit_3M,0.031,0.000025
2,2026-10-27,0.501370,0.983994,0.032183,deposit_6M,0.032,0.000049
3,2027-01-27,0.753425,0.975765,0.032563,fra_6M_9M,0.033,0.000074
4,2027-04-27,1.000000,0.967541,0.032998,fra_9M_12M,0.034,0.000097
5,2028-04-27,2.002740,0.933465,0.034379,swap_2Y,0.035,0.000187
6,2029-04-27,3.002740,0.899193,0.035387,swap_3Y,0.036,0.000270
7,2031-04-28,5.005479,0.833438,0.036399,swap_5Y,0.037,0.000417


,instrument_type,tenor,quote_rate,model_rate,rate_error_bps,abs_price_error
0,deposit,1M,0.030,0.030,-6.383782e-12,0.000000e+00
1,deposit,3M,0.031,0.031,1.734723e-12,0.000000e+00
2,deposit,6M,0.032,0.032,-2.775558e-13,0.000000e+00
3,fra,6Mx9M,0.033,0.033,-1.595946e-12,0.000000e+00
4,fra,9Mx12M,0.034,0.034,-1.942890e-12,0.000000e+00
5,swap,2Y,0.035,0.035,-4.163336e-13,0.000000e+00
6,swap,3Y,0.036,0.036,2.775558e-13,1.110223e-16
7,swap,5Y,0.037,0.037,-2.177425e-10,9.814372e-14


{'positive_discount_factors': True, 'non_increasing_discount_factors': True, 'min_discount_factor': 0.8334376269921968, 'max_discount_factor': 0.9975062344139651, 'point_count': 8.0, 'min_maturity': 0.0821917808219178, 'max_maturity': 5.005479452054795, 'max_abs_rate_error_bps': 2.1774249070460883e-10, 'max_abs_price_error': 9.814371537686384e-14, 'rmse_rate_error_bps': 7.702451190574092e-11}
Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_bootstrap_points.csv
Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_bootstrap_instrument_checks.csv


,section,check,observed,expected,status,details
5,rates,bootstrap_point_count,8,>= 6,PASS,
6,rates,discount_factors_positive,0.8334376269921968,> 0,PASS,
7,rates,discount_factors_decreasing,"[0.9975062344139651, 0.9922248160938865, 0.983...",monotonic decreasing,PASS,
8,rates,bootstrap_repricing_error_bps,2.1774249070460883e-10,< 5 bps,PASS,
9,rates,curve_static_arbitrage,"{'positive_discount_factors': True, 'non_incre...",positive discount factors,PASS,
10,rates,zero_coupon_price_positive,93.35562574771204,> 0,PASS,
11,rates,historical_curve_builds,ValueError: No rate curve data found for count...,no exception,WARN,"Traceback (most recent call last):\n File ""C:..."


## 4. QA volatilité implicite

On choisit automatiquement un sous-jacent avec assez de quotes exploitables, puis on calibre les volatilités implicites depuis les prix marché. Cette étape évite de mélanger plusieurs sous-jacents dans une seule surface.

Commentaires de données :

- les IV du fichier options sont absentes/non exploitables ; elles sont donc inversées depuis les prix Black-Scholes ;
- le panel est court terme et bruité, donc on filtre les spreads bid/ask, les maturités et la moneyness ;
- la sélection du sous-jacent est basée sur la stabilité de calibration/repricing, pas sur le simple nombre de lignes.

In [18]:
calibrated_quotes = pd.DataFrame()
iv_result = None
selected_underlying = None
underlying_selection_table = pd.DataFrame()


def prepare_vol_calibration_universe(option_quotes: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Nettoyage marche avant IV/SVI: un peu plus strict que le loader brut."""
    if option_quotes.empty:
        return pd.DataFrame(), pd.DataFrame()

    panel = clean_option_panel(option_quotes, max_bid_ask_spread_ratio=0.60)
    report: list[dict[str, Any]] = []

    def add_step(name: str, frame: pd.DataFrame) -> None:
        report.append(
            {
                "step": name,
                "rows": len(frame),
                "underlyings": int(frame["underlying"].nunique()) if "underlying" in frame.columns and len(frame) else 0,
                "min_maturity": frame["time_to_maturity_years"].min() if len(frame) else np.nan,
                "max_maturity": frame["time_to_maturity_years"].max() if len(frame) else np.nan,
                "min_moneyness": frame["moneyness"].min() if len(frame) else np.nan,
                "max_moneyness": frame["moneyness"].max() if len(frame) else np.nan,
            }
        )

    add_step("clean_option_panel", panel)

    panel = panel[
        panel["time_to_maturity_years"].between(14.0 / 365.25, 2.5)
        & panel["moneyness"].between(0.70, 1.30)
    ].copy()
    add_step("maturity_moneyness_filter", panel)

    if {"bid", "ask", "market_price"}.issubset(panel.columns):
        panel["bid_ask_spread_ratio"] = (panel["ask"] - panel["bid"]) / panel["market_price"]
        panel = panel[
            panel["bid_ask_spread_ratio"].isna()
            | (panel["bid_ask_spread_ratio"] <= 0.50)
        ].copy()
        add_step("bid_ask_spread_filter", panel)

    return panel.reset_index(drop=True), pd.DataFrame(report)


def calibrate_iv_relaxed(quotes: pd.DataFrame, *, rate: float, dividend_yield: float) -> tuple[pd.DataFrame, Any]:
    """Fallback IV calibration with a looser spread filter for noisy free data."""
    panel = clean_option_panel(
        quotes,
        max_bid_ask_spread_ratio=1.50,
        min_maturity_years=1.0 / 365.25,
        min_price=1e-4,
    )
    if panel.empty:
        return panel, None

    panel["implied_vol"] = panel.apply(
        lambda row: implied_volatility_from_price(
            option_type=str(row["option_type"]),
            market_price=float(row["market_price"]),
            spot=float(row["underlying_price"]),
            strike=float(row["strike"]),
            maturity=float(row["time_to_maturity_years"]),
            rate=rate,
            dividend_yield=dividend_yield,
        ),
        axis=1,
    )
    panel = panel[np.isfinite(panel["implied_vol"]) & (panel["implied_vol"] > 0.0)].copy()
    if panel.empty:
        return panel, None

    panel["repriced"] = panel.apply(
        lambda row: black_scholes_price_and_greeks(
            option_type=str(row["option_type"]),
            spot=float(row["underlying_price"]),
            strike=float(row["strike"]),
            maturity=float(row["time_to_maturity_years"]),
            rate=rate,
            volatility=float(row["implied_vol"]),
            dividend_yield=dividend_yield,
        ).price,
        axis=1,
    )
    panel["calibration_error"] = panel["repriced"] - panel["market_price"]
    return panel.reset_index(drop=True), None


def _candidate_maturity_grid(quotes: pd.DataFrame) -> np.ndarray:
    maturities = np.array(sorted(quotes["time_to_maturity_years"].dropna().unique()), dtype=float)
    if len(maturities) > 6:
        maturities = np.quantile(maturities, np.linspace(0.0, 1.0, 6))
    return maturities[maturities > 0]


def choose_surface_underlying(calibrated_panel: pd.DataFrame) -> tuple[str | None, pd.DataFrame]:
    """Choose the underlying with the most stable one-name SVI/SSVI surface."""
    if calibrated_panel.empty or "underlying" not in calibrated_panel.columns:
        return None, pd.DataFrame()

    thresholds = MarketErrorThresholds(
        abs_price_mae=0.75,
        abs_price_rmse=1.25,
        abs_price_max=8.00,
        abs_relative_price_mae=0.30,
        abs_vol_mae=0.08,
    )
    rows: list[dict[str, Any]] = []

    for underlying, group in calibrated_panel.groupby("underlying", observed=False):
        group = group.copy()
        if len(group) < 80 or group["time_to_maturity_years"].nunique() < 2:
            continue

        row: dict[str, Any] = {"underlying": str(underlying), "rows": len(group)}

        try:
            svi = SVIVolSurface.fit_from_quotes(group, min_points_per_slice=5)
            svi_diag = svi.diagnostics(log_moneyness_grid=np.linspace(-0.50, 0.50, 81))
            svi_validation = reprice_vanilla_market_quotes(
                group,
                svi,
                rate=QA_RATE,
                dividend_yield=QA_DIVIDEND_YIELD,
                thresholds=thresholds,
            )
            svi_summary = svi_validation.summary.iloc[0]
            row.update(
                svi_fit=True,
                svi_butterfly_ok=bool(svi_diag.get("butterfly_arbitrage_free", False)),
                svi_calendar_ok=bool(svi_diag.get("calendar_arbitrage_free", False)),
                svi_within_thresholds=bool(svi_summary["within_thresholds"]),
                svi_rmse=float(svi_summary["rmse"]),
                svi_relative_mae=float(svi_summary["mean_abs_relative_error"]),
            )
        except Exception as exc:
            row.update(
                svi_fit=False,
                svi_butterfly_ok=False,
                svi_calendar_ok=False,
                svi_within_thresholds=False,
                svi_rmse=np.inf,
                svi_relative_mae=np.inf,
                svi_error=f"{type(exc).__name__}: {exc}",
            )

        try:
            ssvi = SSVIVolSurface.fit_from_quotes(group, max_multistarts=6)
            ssvi_diag = ssvi.diagnostics(
                maturity_grid=_candidate_maturity_grid(group),
                log_moneyness_grid=np.linspace(-0.50, 0.50, 81),
            )
            ssvi_validation = reprice_vanilla_market_quotes(
                group,
                ssvi,
                rate=QA_RATE,
                dividend_yield=QA_DIVIDEND_YIELD,
                thresholds=thresholds,
            )
            ssvi_summary = ssvi_validation.summary.iloc[0]
            row.update(
                ssvi_fit=True,
                ssvi_butterfly_ok=bool(ssvi_diag.get("butterfly_arbitrage_free", False)),
                ssvi_calendar_ok=bool(ssvi_diag.get("calendar_arbitrage_free", False)),
                ssvi_within_thresholds=bool(ssvi_summary["within_thresholds"]),
                ssvi_rmse=float(ssvi_summary["rmse"]),
                ssvi_relative_mae=float(ssvi_summary["mean_abs_relative_error"]),
            )
        except Exception as exc:
            row.update(
                ssvi_fit=False,
                ssvi_butterfly_ok=False,
                ssvi_calendar_ok=False,
                ssvi_within_thresholds=False,
                ssvi_rmse=np.inf,
                ssvi_relative_mae=np.inf,
                ssvi_error=f"{type(exc).__name__}: {exc}",
            )

        rows.append(row)

    table = pd.DataFrame(rows)
    if table.empty:
        counts = calibrated_panel["underlying"].value_counts(dropna=True)
        return (None if counts.empty else str(counts.index[0])), table

    table = table.sort_values(
        [
            "ssvi_within_thresholds",
            "ssvi_calendar_ok",
            "ssvi_butterfly_ok",
            "svi_within_thresholds",
            "ssvi_rmse",
            "svi_rmse",
            "rows",
        ],
        ascending=[False, False, False, False, True, True, False],
        ignore_index=True,
    )
    return str(table.iloc[0]["underlying"]), table


try:
    # Limite des données options : IV absentes et panel court terme/bruité.
    # Le nettoyage bid/ask, maturité et moneyness fait partie de la validation.
    vol_universe, vol_filter_report = prepare_vol_calibration_universe(option_quotes)
    display(vol_filter_report)
    save_frame(vol_filter_report, "qa_vol_filter_report.csv")

    calibrated_quotes, iv_result = calibrate_implied_vol_panel(
        vol_universe,
        rate=QA_RATE,
        dividend_yield=QA_DIVIDEND_YIELD,
        drop_outliers=True,
    )

    if calibrated_quotes.empty:
        calibrated_quotes, iv_result = calibrate_iv_relaxed(
            option_quotes,
            rate=QA_RATE,
            dividend_yield=QA_DIVIDEND_YIELD,
        )
        record("vol_iv", "strict_calibration_fallback", "relaxed filter used", "strict calibration non-empty", "WARN")

    # Une surface SVI/SSVI doit être mono-sous-jacent. On choisit la meilleure
    # qualité de calibration/repricing plutôt que le ticker avec le plus de lignes.
    selected_underlying, underlying_selection_table = choose_surface_underlying(calibrated_quotes)
    if selected_underlying is None:
        raise ValueError("No option underlying available for a one-name volatility surface")

    display(underlying_selection_table)
    save_frame(underlying_selection_table, "qa_underlying_surface_selection.csv")

    calibrated_quotes = calibrated_quotes[
        calibrated_quotes["underlying"].astype(str) == selected_underlying
    ].copy().reset_index(drop=True)

    if len(calibrated_quotes) > MAX_VOL_QUOTES:
        calibrated_quotes = calibrated_quotes.sample(MAX_VOL_QUOTES, random_state=RANDOM_SEED).sort_values(
            ["time_to_maturity_years", "log_moneyness"]
        ).reset_index(drop=True)
        record("vol_iv", "calibration_sample_cap", len(calibrated_quotes), f"<= {MAX_VOL_QUOTES}", "WARN", "Dataset capped for notebook runtime.")

    display(calibrated_quotes.head())
    display(calibrated_quotes[["time_to_maturity_years", "log_moneyness", "implied_vol", "market_price", "calibration_error"]].describe())

    check_true("vol_iv", "selected_underlying", selected_underlying is not None, observed=selected_underlying, expected="non-null")
    check_true("vol_iv", "one_underlying_surface_input", calibrated_quotes["underlying"].nunique() == 1, observed=selected_underlying, expected="single underlying")
    check_true("vol_iv", "calibrated_quote_count", len(calibrated_quotes) >= 12, observed=len(calibrated_quotes), expected=">= 12", fail_if_false=True)
    check_true("vol_iv", "iv_positive", bool((calibrated_quotes["implied_vol"] > 0).all()), observed=float(calibrated_quotes["implied_vol"].min()) if len(calibrated_quotes) else np.nan, expected="> 0")
    check_true("vol_iv", "iv_repricing_error", calibrated_quotes["calibration_error"].abs().median() < 1e-5, observed=float(calibrated_quotes["calibration_error"].abs().median()), expected="< 1e-5")

    save_frame(calibrated_quotes, "qa_calibrated_option_quotes.csv")
except Exception as exc:
    record_exception("vol_iv", "implied_vol_calibration", exc, fail=True)

display_qa("vol_iv")

,step,rows,underlyings,min_maturity,max_maturity,min_moneyness,max_moneyness
0,clean_option_panel,6704,10,0.03833,0.049281,0.007488,3.071768
1,maturity_moneyness_filter,4324,10,0.03833,0.049281,0.700762,1.299792
2,bid_ask_spread_filter,4265,10,0.03833,0.049281,0.700762,1.299792


Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_vol_filter_report.csv


,underlying,rows,svi_fit,svi_butterfly_ok,svi_calendar_ok,svi_within_thresholds,svi_rmse,svi_relative_mae,ssvi_fit,ssvi_butterfly_ok,ssvi_calendar_ok,ssvi_within_thresholds,ssvi_rmse,ssvi_relative_mae,ssvi_error
0,NVDA,351,True,False,False,True,0.157758,0.176773,True,True,True,True,0.333835,0.203477,NaN
1,TSLA,567,True,True,False,True,0.288130,0.163464,True,True,True,True,0.531313,0.195721,NaN
2,AAPL,348,True,False,False,False,0.288869,0.344862,True,True,True,False,0.432487,0.551569,NaN
3,GOOGL,415,True,True,False,False,0.315622,0.329560,True,True,True,False,0.486930,0.420009,NaN
4,AMZN,324,True,False,False,False,0.269223,0.380650,True,True,True,False,0.530366,0.497553,NaN
5,MSFT,489,True,False,False,False,0.417475,0.461197,True,True,True,False,0.619962,0.553657,NaN
6,META,789,True,True,False,False,0.602666,0.414440,True,True,True,False,1.084681,0.513213,NaN
7,UNH,321,True,False,False,False,1.253873,0.900183,True,True,True,False,1.356456,0.945964,NaN
8,BRKB,351,True,False,False,False,1.175687,0.411954,True,True,True,False,1.990472,0.796566,NaN
9,JNJ,209,True,False,False,True,0.585698,0.099280,False,False,False,False,inf,inf,RuntimeError: SSVI calibration failed: ABNORMAL:


Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_underlying_surface_selection.csv


,contract_symbol,ticker,underlying,option_type,valuation_date,expiration_date,time_to_maturity_days,time_to_maturity_years,strike,bid,mid,ask,last,underlying_price,implied_volatility,intrinsic_value,extrinsic_value,bid_size,ask_size,open_interest,volume,in_the_money,first_traded_at,last_updated_at,delta,gamma,theta,vega,rho,market_price,moneyness,log_moneyness,bid_ask_spread_ratio,implied_vol,repriced,calibration_error
0,NVDA260320C00130000,NVDA,NVDA,call,2026-03-02,2026-03-20,18,0.049281,130.0,52.65,52.975,53.3,53.05,182.48,<NA>,52.48,0.495,52,52,25366,208,True,2025-01-15 14:30:00,2026-03-02 21:00:00,<NA>,<NA>,<NA>,<NA>,<NA>,52.975,0.712407,-0.339106,0.01227,0.806992,52.975,-0.0
1,NVDA260320C00135000,NVDA,NVDA,call,2026-03-02,2026-03-20,18,0.049281,135.0,47.8,48.075,48.35,47.96,182.48,<NA>,47.48,0.595,12,26,26637,50,True,2025-01-15 14:30:00,2026-03-02 21:00:00,<NA>,<NA>,<NA>,<NA>,<NA>,48.075,0.739807,-0.301366,0.01144,0.765036,48.075,-0.0
2,NVDA260320C00140000,NVDA,NVDA,call,2026-03-02,2026-03-20,18,0.049281,140.0,42.85,43.175,43.5,43.35,182.48,<NA>,42.48,0.695,52,82,26154,1793,True,2025-01-15 14:30:00,2026-03-02 21:00:00,<NA>,<NA>,<NA>,<NA>,<NA>,43.175,0.767207,-0.264998,0.015055,0.715368,43.175,-0.0
3,NVDA260320C00145000,NVDA,NVDA,call,2026-03-02,2026-03-20,18,0.049281,145.0,38.0,38.325,38.65,38.1,182.48,<NA>,37.48,0.845,52,25,26515,43,True,2025-01-15 14:30:00,2026-03-02 21:00:00,<NA>,<NA>,<NA>,<NA>,<NA>,38.325,0.794608,-0.229907,0.01696,0.672031,38.325,-0.0
4,NVDA260320C00150000,NVDA,NVDA,call,2026-03-02,2026-03-20,18,0.049281,150.0,33.45,33.625,33.8,33.45,182.48,<NA>,32.48,1.145,12,52,43032,420,True,2025-01-15 14:30:00,2026-03-02 21:00:00,<NA>,<NA>,<NA>,<NA>,<NA>,33.625,0.822008,-0.196005,0.010409,0.647915,33.625,-0.0


,time_to_maturity_years,log_moneyness,implied_vol,market_price,calibration_error
count,351.0,351.0,351.000000,351.0,351.0
mean,0.043275,-0.011303,0.561219,13.371254,-0.0
std,0.003999,0.162811,0.143548,15.110192,0.0
min,0.03833,-0.352458,0.403429,0.035,-0.0
25%,0.03833,-0.136758,0.448420,0.9775,-0.0
50%,0.043806,0.010651,0.504463,6.35,-0.0
75%,0.046543,0.123789,0.653355,22.625,0.0
max,0.049281,0.257308,0.956491,53.675,0.0


Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_calibrated_option_quotes.csv


,section,check,observed,expected,status,details
12,vol_iv,selected_underlying,NVDA,non-null,PASS,
13,vol_iv,one_underlying_surface_input,NVDA,single underlying,PASS,
14,vol_iv,calibrated_quote_count,351,>= 12,PASS,
15,vol_iv,iv_positive,0.4034294328569635,> 0,PASS,
16,vol_iv,iv_repricing_error,1.6442402994698568e-13,< 1e-5,PASS,


## 5. QA SVI / SSVI et contrôles d'arbitrage

La règle de décision utilisée ici est :

- `SSVI` est le modèle principal si ses contrôles `calendar` et `butterfly` passent ;
- `SVI` brut est conservé comme benchmark pédagogique, mais un échec d'arbitrage est classé en `WARN` plutôt qu'en `FAIL` sur données marché bruitées.

Limite à documenter : avec très peu de maturités et des prix bruités, le SVI calibré tranche par tranche peut créer une non-monotonicité de variance totale entre maturités. Le contrôle reste utile : il justifie de privilégier SSVI comme surface régularisée.

In [19]:
svi_surface = None
ssvi_surface = None
selected_surface = None
selected_surface_name = None
svi_diag = {}
ssvi_diag = {}

try:
    if calibrated_quotes.empty:
        raise ValueError("No calibrated quotes available for SVI/SSVI")

    maturity_grid = np.unique(np.quantile(calibrated_quotes["time_to_maturity_years"], [0.05, 0.25, 0.50, 0.75, 0.95]))
    maturity_grid = maturity_grid[maturity_grid > 0]
    if len(maturity_grid) < 2:
        maturity_grid = np.array(sorted(calibrated_quotes["time_to_maturity_years"].unique()))
    if len(maturity_grid) < 2:
        maturity_grid = np.array([float(calibrated_quotes["time_to_maturity_years"].median()), float(calibrated_quotes["time_to_maturity_years"].median()) * 1.5])

    k_low, k_high = calibrated_quotes["log_moneyness"].quantile([0.05, 0.95]).astype(float)
    k_low = max(k_low, -0.70)
    k_high = min(k_high, 0.70)
    if k_high <= k_low:
        k_low, k_high = -0.30, 0.30
    log_moneyness_grid = np.linspace(k_low, k_high, 101)

    # SVI brut par maturité.
    try:
        svi_surface = SVIVolSurface.fit_from_quotes(calibrated_quotes, min_points_per_slice=5)
        svi_diag = svi_surface.diagnostics(log_moneyness_grid=log_moneyness_grid)
        print("SVI diagnostics:", svi_diag)
        check_true("vol_svi", "svi_fit_success", True, observed=svi_surface.calibration.parameters, expected="fit succeeds")
        check_true("vol_svi", "svi_butterfly_arbitrage_free", bool(svi_diag.get("butterfly_arbitrage_free", False)), observed=svi_diag, expected=True, fail_if_false=False)
        check_true("vol_svi", "svi_calendar_arbitrage_free", bool(svi_diag.get("calendar_arbitrage_free", False)), observed=svi_diag, expected=True, fail_if_false=False)
    except Exception as exc:
        record_exception("vol_svi", "svi_fit_success", exc, fail=False)

    # SSVI global.
    try:
        ssvi_surface = SSVIVolSurface.fit_from_quotes(calibrated_quotes, max_multistarts=6)
        ssvi_diag = ssvi_surface.diagnostics(maturity_grid=maturity_grid, log_moneyness_grid=log_moneyness_grid)
        print("SSVI diagnostics:", ssvi_diag)
        check_true("vol_ssvi", "ssvi_fit_success", True, observed=ssvi_surface.calibration.parameters, expected="fit succeeds")
        check_true("vol_ssvi", "ssvi_butterfly_arbitrage_free", bool(ssvi_diag.get("butterfly_arbitrage_free", False)), observed=ssvi_diag, expected=True, fail_if_false=False)
        check_true("vol_ssvi", "ssvi_calendar_arbitrage_free", bool(ssvi_diag.get("calendar_arbitrage_free", False)), observed=ssvi_diag, expected=True, fail_if_false=False)
    except Exception as exc:
        record_exception("vol_ssvi", "ssvi_fit_success", exc, fail=False)

    if ssvi_surface is not None and ssvi_diag.get("butterfly_arbitrage_free", False) and ssvi_diag.get("calendar_arbitrage_free", False):
        selected_surface = ssvi_surface
        selected_surface_name = "SSVI"
    elif ssvi_surface is not None:
        selected_surface = ssvi_surface
        selected_surface_name = "SSVI_WITH_WARNINGS"
    elif svi_surface is not None:
        selected_surface = svi_surface
        selected_surface_name = "SVI_WITH_WARNINGS"
    else:
        selected_surface = None
        selected_surface_name = None

    check_true("vol_surface_selection", "selected_surface_available", selected_surface is not None, observed=selected_surface_name, expected="SSVI preferred", fail_if_false=True)

    diagnostics_frame = pd.DataFrame(
        [
            {"surface": "SVI", **svi_diag},
            {"surface": "SSVI", **ssvi_diag},
        ]
    )
    display(diagnostics_frame)
    save_frame(diagnostics_frame, "qa_vol_surface_diagnostics.csv")
except Exception as exc:
    record_exception("vol_surface_selection", "surface_workflow", exc, fail=True)

display_qa("vol_svi")
display_qa("vol_ssvi")
display_qa("vol_surface_selection")

SVI diagnostics: {'butterfly_arbitrage_free': True, 'calendar_arbitrage_free': False, 'min_butterfly_convexity': 0.013889932173327584, 'max_total_variance': 0.030541013808887924, 'min_total_variance': 0.006509955375500092}
SSVI diagnostics: {'butterfly_arbitrage_free': True, 'calendar_arbitrage_free': True, 'min_butterfly_convexity': 0.14727801785746575, 'max_total_variance': 0.030921285228763865, 'min_total_variance': 0.007202634983409681}


,surface,butterfly_arbitrage_free,calendar_arbitrage_free,min_butterfly_convexity,max_total_variance,min_total_variance
0,SVI,True,False,0.013890,0.030541,0.006510
1,SSVI,True,True,0.147278,0.030921,0.007203


Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_vol_surface_diagnostics.csv


,section,check,observed,expected,status,details
17,vol_svi,svi_fit_success,"{'slice_count': 5.0, 'quote_count': 351.0, 'me...",fit succeeds,PASS,
18,vol_svi,svi_butterfly_arbitrage_free,"{'butterfly_arbitrage_free': True, 'calendar_a...",True,PASS,
19,vol_svi,svi_calendar_arbitrage_free,"{'butterfly_arbitrage_free': True, 'calendar_a...",True,WARN,


,section,check,observed,expected,status,details
20,vol_ssvi,ssvi_fit_success,"{'v0': 0.20882661150314485, 'v_inf': 0.2283173...",fit succeeds,PASS,
21,vol_ssvi,ssvi_butterfly_arbitrage_free,"{'butterfly_arbitrage_free': True, 'calendar_a...",True,PASS,
22,vol_ssvi,ssvi_calendar_arbitrage_free,"{'butterfly_arbitrage_free': True, 'calendar_a...",True,PASS,


,section,check,observed,expected,status,details
23,vol_surface_selection,selected_surface_available,SSVI,SSVI preferred,PASS,


## 6. QA validation marché : repricing de vanilles

On mesure les erreurs modèle/prix sur un jeu de test quand c'est possible. Si le split train/test n'est pas exploitable, la validation revient en in-sample avec un `WARN`.

Limite à documenter : ce test est une validation pratique sur le panel disponible, pas un backtest marché complet. Le fichier options est court terme et quasi mono-horizon ; on valide donc la cohérence prix/modèle sur une coupe de marché plutôt qu'une robustesse multi-date.

In [20]:
validation_result = None
validation_quotes = pd.DataFrame()

try:
    if selected_surface is None or calibrated_quotes.empty:
        raise ValueError("No selected volatility surface or calibrated quote panel")

    thresholds = MarketErrorThresholds(
        abs_price_mae=0.75,
        abs_price_rmse=1.25,
        abs_price_max=8.00,
        abs_relative_price_mae=0.30,
        abs_vol_mae=0.08,
    )

    # Split déterministe. Pour un vrai rapport, privilégier un split par maturité ou par date si plusieurs dates existent.
    shuffled = calibrated_quotes.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    train_size = int(max(12, min(len(shuffled) - 5, round(TRAIN_FRACTION * len(shuffled)))))
    if train_size <= 0 or train_size >= len(shuffled):
        validation_quotes = calibrated_quotes.copy()
        record("market_validation", "train_test_split", "in-sample", "out-of-sample preferred", "WARN", "Panel too small for a robust split.")
    else:
        train_quotes = shuffled.iloc[:train_size].copy()
        test_quotes = shuffled.iloc[train_size:].copy()
        validation_quotes = test_quotes.copy()
        record("market_validation", "train_test_split", f"train={len(train_quotes)}, test={len(test_quotes)}", "non-empty test", "PASS")

        # Pour une vraie validation out-of-sample, on essaie de recalibrer SSVI sur train.
        # Si la calibration train échoue, on garde la surface globale et on conserve le WARN.
        try:
            train_surface = SSVIVolSurface.fit_from_quotes(train_quotes, max_multistarts=4)
            selected_surface_for_validation = train_surface
            record("market_validation", "train_surface_fit", "SSVI train", "fit succeeds", "PASS")
        except Exception as exc:
            selected_surface_for_validation = selected_surface
            record("market_validation", "train_surface_fit", f"fallback global surface: {type(exc).__name__}", "SSVI train fit", "WARN", str(exc))
    if 'selected_surface_for_validation' not in locals():
        selected_surface_for_validation = selected_surface

    validation_result = reprice_vanilla_market_quotes(
        validation_quotes,
        selected_surface_for_validation,
        rate=QA_RATE,
        dividend_yield=QA_DIVIDEND_YIELD,
        thresholds=thresholds,
    )

    display(validation_result.summary)
    display(validation_result.by_maturity.head(20))
    display(validation_result.by_moneyness_bucket)

    row = validation_result.summary.iloc[0]
    check_true("market_validation", "repricing_quote_count", int(row["quote_count"]) > 0, observed=int(row["quote_count"]), expected="> 0")
    check_true("market_validation", "repricing_within_thresholds", bool(row["within_thresholds"]), observed=row.to_dict(), expected="within thresholds", fail_if_false=False)

    save_frame(validation_result.summary, "qa_repricing_summary.csv")
    save_frame(validation_result.by_maturity, "qa_repricing_by_maturity.csv")
    save_frame(validation_result.by_moneyness_bucket, "qa_repricing_by_moneyness.csv")
    save_frame(validation_result.line_errors, "qa_repricing_line_errors.csv")
except Exception as exc:
    record_exception("market_validation", "vanilla_repricing", exc, fail=True)

display_qa("market_validation")

,quote_count,mean_error,mae,rmse,max_abs_error,mean_abs_relative_error,mean_abs_vol_error,threshold_abs_price_mae,threshold_abs_price_rmse,threshold_abs_price_max,threshold_abs_relative_price_mae,threshold_abs_vol_mae,within_thresholds
0,70.0,-0.11099,0.256208,0.377227,1.420885,0.118817,0.052024,0.75,1.25,8.0,0.3,0.08,True


,maturity_bucket,quote_count,mean_error,mae,max_abs_error,mean_abs_relative_error,mean_abs_vol_error,rmse
0,0.04,47,-0.165712,0.329617,1.420885,0.125395,0.069940,0.452627
1,0.05,23,0.000834,0.106199,0.230141,0.105377,0.015411,0.120159


,moneyness_bucket,quote_count,mean_error,mae,max_abs_error,mean_abs_relative_error,mean_abs_vol_error,rmse
0,deep_otm_put/itm_call,12,-0.069795,0.215197,0.565565,0.172948,0.067053,0.256984
1,otm_put/itm_call,13,-0.06188,0.214591,0.549268,0.061378,0.031456,0.249955
2,atm,16,-0.092689,0.23152,1.057294,0.030411,0.016972,0.366923
3,otm_call/itm_put,22,0.025873,0.156083,0.33431,0.195882,0.027141,0.182375
4,deep_otm_call/itm_put,7,-0.744788,0.77491,1.420885,0.092565,0.222775,0.884023


Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_repricing_summary.csv
Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_repricing_by_maturity.csv
Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_repricing_by_moneyness.csv
Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_repricing_line_errors.csv


,section,check,observed,expected,status,details
24,market_validation,train_test_split,"train=281, test=70",non-empty test,PASS,
25,market_validation,train_surface_fit,SSVI train,fit succeeds,PASS,
26,market_validation,repricing_quote_count,70,> 0,PASS,
27,market_validation,repricing_within_thresholds,"{'quote_count': 70.0, 'mean_error': -0.1109901...",within thresholds,PASS,


## 7. QA portefeuille et agrégations

Cette section vérifie que l'inventaire est ingéré, que les produits supportés sont valorisés, et que les agrégations sont cohérentes. Les produits non supportés sont attendus tant que swaps/autocalls ne sont pas implémentés complètement.

Pour les fondations marché, l'objectif est que les lignes non supportées soient identifiées explicitement. Leur pricing événementiel complet relève du prochain jalon produit/Monte Carlo.

In [21]:
portfolio_result = None

try:
    if inventory_all.empty:
        raise ValueError("No inventory data available")

    # Contexte de marché simple pour QA. En amélioration, remplacer par surface/courbe calibrée et spots par sous-jacent.
    spot_by_underlying = {}
    if non_empty_frame(option_quotes) and {"underlying", "underlying_price"}.issubset(option_quotes.columns):
        latest_spots = (
            option_quotes.dropna(subset=["underlying", "underlying_price"])
            .groupby("underlying")["underlying_price"]
            .median()
            .to_dict()
        )
        spot_by_underlying = {str(k): float(v) for k, v in latest_spots.items() if np.isfinite(v)}

    context = PortfolioMarketContext(
        default_spot=100.0,
        rate=QA_RATE,
        volatility=0.20,
        dividend_yield=QA_DIVIDEND_YIELD,
        spot_by_underlying=spot_by_underlying,
    )
    engine = PortfolioValuationEngine()
    portfolio_result = engine.value_inventory(inventory_all, market=context)

    display(portfolio_result.line_valuations.head(30))
    display(portfolio_result.by_product)
    display(portfolio_result.by_portfolio)

    supported = portfolio_result.line_valuations[portfolio_result.line_valuations["status"] == "supported"]
    unsupported = portfolio_result.line_valuations[portfolio_result.line_valuations["status"] != "supported"]

    total_lines = float(supported["price"].sum()) if not supported.empty else 0.0
    total_portfolio = float(portfolio_result.by_portfolio["price"].sum()) if not portfolio_result.by_portfolio.empty else 0.0

    check_true("portfolio", "inventory_line_count", len(inventory_all) > 0, observed=len(inventory_all), expected="> 0")
    check_true("portfolio", "supported_line_count", len(supported) > 0, observed=len(supported), expected="> 0", fail_if_false=False)
    check_true("portfolio", "unsupported_lines_flagged", True, observed=len(unsupported), expected="flagged, not silently priced")
    check_true("portfolio", "aggregation_total_consistency", abs(total_lines - total_portfolio) < 1e-8, observed=(total_lines, total_portfolio), expected="line sum == portfolio sum")

    if len(unsupported) > 0:
        record(
            "portfolio",
            "unsupported_product_scope",
            unsupported[["source_sheet", "product_type", "error_message"]].drop_duplicates().to_dict("records"),
            "0 unsupported for full scope",
            "WARN",
            "À traiter si tu veux couvrir swaps/autocalls dans la version finale.",
        )

    save_frame(portfolio_result.line_valuations, "qa_portfolio_line_valuations.csv")
    save_frame(portfolio_result.by_product, "qa_portfolio_by_product.csv")
    save_frame(portfolio_result.by_portfolio, "qa_portfolio_by_portfolio.csv")
except Exception as exc:
    record_exception("portfolio", "portfolio_valuation", exc, fail=False)

display_qa("portfolio")

,line_index,portfolio,product_id,source_sheet,underlying,quantity,product_type,status,price,delta,gamma,vega,theta,rho,maturity_years,strike,error_message
0,0,default_portfolio,<NA>,swaps,<NA>,1.0,<na>,unsupported,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN,"Unsupported product row: source_sheet='swaps',..."
1,1,default_portfolio,<NA>,swaps,<NA>,1.0,<na>,unsupported,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN,"Unsupported product row: source_sheet='swaps',..."
2,2,default_portfolio,<NA>,swaps,<NA>,1.0,<na>,unsupported,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN,"Unsupported product row: source_sheet='swaps',..."
3,3,default_portfolio,<NA>,swaps,<NA>,1.0,<na>,unsupported,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN,"Unsupported product row: source_sheet='swaps',..."
4,4,default_portfolio,<NA>,swaps,<NA>,1.0,<na>,unsupported,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN,"Unsupported product row: source_sheet='swaps',..."
5,5,default_portfolio,<NA>,options,AAPL,100000.0,call_spread,supported,2.231887e+06,7.958501e+04,-8.177470e+02,-9.566327e+05,5.673062e+05,1.583802e+06,0.084873,260.000000,
6,6,default_portfolio,<NA>,options,AAPL,250000.0,butterfly,supported,3.181726e+06,-2.327879e+04,6.768050e+01,3.141472e+05,1.854998e+05,-3.129428e+06,0.336756,226.666667,
7,7,default_portfolio,<NA>,options,AAPL,100000.0,vanilla_call,supported,1.476980e+06,5.902487e+04,1.275715e+03,5.921387e+06,-2.178910e+06,4.720718e+06,0.336756,260.000000,
8,8,default_portfolio,<NA>,options,AAPL,100000.0,call_spread,supported,8.271608e+05,2.494703e+04,7.207339e+01,3.345374e+05,-2.709993e+05,1.926893e+06,0.336756,270.000000,
9,9,default_portfolio,<NA>,options,AAPL,100000.0,vanilla_put,supported,9.636328e+05,-4.097513e+04,1.275715e+03,5.921387e+06,-1.406750e+06,-3.946919e+06,0.336756,260.000000,


,product_type,price,delta,gamma,vega,theta,rho,line_count
0,butterfly,3.181726e+06,-2.327879e+04,6.768050e+01,3.141472e+05,1.854998e+05,-3.129428e+06,1
1,call_spread,3.059047e+06,1.045320e+05,-7.456736e+02,-6.220953e+05,2.963070e+05,3.510696e+06,2
2,put_spread,2.850159e-12,-1.212818e-13,4.246745e-15,3.445555e-11,-4.812766e-12,-2.041930e-11,1
3,vanilla_call,1.476980e+06,5.902487e+04,1.275715e+03,5.921387e+06,-2.178910e+06,4.720718e+06,1
4,vanilla_put,9.636328e+05,-4.097513e+04,1.275715e+03,5.921387e+06,-1.406750e+06,-3.946919e+06,1


,portfolio,price,delta,gamma,vega,theta,rho,line_count
0,default_portfolio,8.681386e+06,99302.97899,1873.43741,1.153483e+07,-3.103853e+06,1.155067e+06,6


Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_portfolio_line_valuations.csv
Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_portfolio_by_product.csv
Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_portfolio_by_portfolio.csv


,section,check,observed,expected,status,details
28,portfolio,inventory_line_count,111,> 0,PASS,
29,portfolio,supported_line_count,6,> 0,PASS,
30,portfolio,unsupported_lines_flagged,105,"flagged, not silently priced",PASS,
31,portfolio,aggregation_total_consistency,"(8681385.71669193, 8681385.71669193)",line sum == portfolio sum,PASS,
32,portfolio,unsupported_product_scope,"[{'source_sheet': 'swaps', 'product_type': '<n...",0 unsupported for full scope,WARN,À traiter si tu veux couvrir swaps/autocalls d...


## 8. Synthèse finale et exports

La table ci-dessous est la base de décision avant clean-up :

- `FAIL` : à corriger avant rendu ;
- `WARN` : acceptable uniquement si justifié dans la documentation ;
- `PASS` : validé par ce notebook.

In [22]:
summary = qa_table()
status_counts = summary["status"].value_counts().reindex(["PASS", "WARN", "FAIL"], fill_value=0)

print("Synthèse QA")
print(status_counts)

display(summary)

save_frame(summary, "qa_status_summary.csv")

# Vue condensée par section.
section_summary = (
    summary.assign(count=1)
    .pivot_table(index="section", columns="status", values="count", aggfunc="sum", fill_value=0)
    .reset_index()
)
for col in ["PASS", "WARN", "FAIL"]:
    if col not in section_summary.columns:
        section_summary[col] = 0
section_summary = section_summary[["section", "PASS", "WARN", "FAIL"]]
display(section_summary)
save_frame(section_summary, "qa_status_by_section.csv")

if status_counts.get("FAIL", 0) > 0:
    display(Markdown("### Décision : ❌ cleanup déconseillé avant correction des FAIL"))
elif status_counts.get("WARN", 0) > 0:
    display(Markdown("### Décision : ⚠️ cleanup possible, mais les WARN doivent être documentés"))
else:
    display(Markdown("### Décision : ✅ projet prêt pour clean-up final"))

Synthèse QA
status
PASS    30
WARN     3
FAIL     0
Name: count, dtype: int64


,section,check,observed,expected,status,details
0,architecture,project_imports,True,True,PASS,Core src imports succeeded.
1,unit_tests,pytest_q,returncode=0,returncode=0,PASS,[32m.[0m[32m.[0m[32m.[0m[32m.[0m[32m....
2,data,rate_curves_loaded,357978,> 0,PASS,
3,data,option_quotes_loaded,10456,> 0,PASS,
4,data,inventory_loaded,111,> 0,PASS,
5,rates,bootstrap_point_count,8,>= 6,PASS,
6,rates,discount_factors_positive,0.8334376269921968,> 0,PASS,
7,rates,discount_factors_decreasing,"[0.9975062344139651, 0.9922248160938865, 0.983...",monotonic decreasing,PASS,
8,rates,bootstrap_repricing_error_bps,2.1774249070460883e-10,< 5 bps,PASS,
9,rates,curve_static_arbitrage,"{'positive_discount_factors': True, 'non_incre...",positive discount factors,PASS,


Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_status_summary.csv


status,section,PASS,WARN,FAIL
0,architecture,1,0,0
1,data,3,0,0
2,market_validation,4,0,0
3,portfolio,4,1,0
4,rates,6,1,0
5,unit_tests,1,0,0
6,vol_iv,5,0,0
7,vol_ssvi,3,0,0
8,vol_surface_selection,1,0,0
9,vol_svi,2,1,0


Exported: C:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Produits Structurés\Projet\structured-products-pricer\reports\qa\qa_status_by_section.csv


### Décision : ⚠️ cleanup possible, mais les WARN doivent être documentés

## 9. Interprétation attendue

À ce stade, l'interprétation normale est la suivante :

- **Taux** : le moteur de bootstrapping dépôt/FRA/swap doit passer sur quotes synthétiques contrôlées. Les données du cours valident en plus le loader de courbes historiques, la sélection pays/date et la cohérence ZC.
- **Volatilité** : SSVI doit être privilégié. SVI brut peut rester en `WARN` si les données marché sont bruitées.
- **Validation marché** : un `WARN` sur les seuils est acceptable uniquement si le panel est court terme / bruité / in-sample. Ajouter ensuite une validation out-of-sample plus stricte si des données multi-dates sont disponibles.
- **Portefeuille** : les produits unsupported doivent être listés explicitement. Pour un rendu complet, choisir entre implémenter swaps/autocalls ou les documenter comme hors périmètre de la version finale.

Conclusion fondations marché : avec les données disponibles, les loaders, courbes ZC, conventions, IV, SSVI et repricing vanille sont validés sans `FAIL`. Les limites restantes sont méthodologiques et correctement tracées en `WARN`.